## 准备数据

In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [10]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [11]:
class myModel:
    def __init__(self):
        self.W1 = tf.Variable(tf.random.normal(shape=[28 * 28, 100], stddev=0.1), name='W1')
        self.b1 = tf.Variable(tf.zeros(shape=[100]), name='b1')
        self.W2 = tf.Variable(tf.random.normal(shape=[100, 10], stddev=0.1), name='W2')
        self.b2 = tf.Variable(tf.zeros(shape=[10]), name='b2')

    def __call__(self, x):
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [12]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1, output_type=tf.int64)
    labels = tf.cast(labels, tf.int64)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [13]:
train_data, test_data = mnist_dataset()

x_train = tf.constant(train_data[0], dtype=tf.float32)
y_train = tf.constant(train_data[1], dtype=tf.int64)
x_test = tf.constant(test_data[0], dtype=tf.float32)
y_test = tf.constant(test_data[1], dtype=tf.int64)

batch_size = 128
epochs = 10
num_train = x_train.shape[0]

for epoch in range(epochs):
    idx = tf.random.shuffle(tf.range(num_train))
    x_train_epoch = tf.gather(x_train, idx)
    y_train_epoch = tf.gather(y_train, idx)

    epoch_loss = []
    epoch_acc = []
    for i in range(0, num_train, batch_size):
        xb = x_train_epoch[i:i + batch_size]
        yb = y_train_epoch[i:i + batch_size]
        loss, accuracy = train_one_step(model, optimizer, xb, yb)
        epoch_loss.append(loss.numpy())
        epoch_acc.append(accuracy.numpy())

    test_loss, test_acc = test(model, x_test, y_test)
    print('epoch', epoch,
          ': train loss', float(np.mean(epoch_loss)),
          '; train accuracy', float(np.mean(epoch_acc)),
          '; test loss', test_loss.numpy(),
          '; test accuracy', test_acc.numpy())

epoch 0 : train loss 0.4186365306377411 ; train accuracy 0.881851851940155 ; test loss 0.21161148 ; test accuracy 0.9401
epoch 1 : train loss 0.1830991506576538 ; train accuracy 0.948238730430603 ; test loss 0.15147416 ; test accuracy 0.9545
epoch 2 : train loss 0.1357601284980774 ; train accuracy 0.9607542753219604 ; test loss 0.12551883 ; test accuracy 0.9622
epoch 3 : train loss 0.10839758813381195 ; train accuracy 0.9684668183326721 ; test loss 0.10939956 ; test accuracy 0.9651
epoch 4 : train loss 0.0882909819483757 ; train accuracy 0.9745302796363831 ; test loss 0.09861184 ; test accuracy 0.9704
epoch 5 : train loss 0.07399829477071762 ; train accuracy 0.9786391854286194 ; test loss 0.09070941 ; test accuracy 0.9723
epoch 6 : train loss 0.06361883878707886 ; train accuracy 0.9819929599761963 ; test loss 0.08854451 ; test accuracy 0.9731
epoch 7 : train loss 0.054704517126083374 ; train accuracy 0.9843361377716064 ; test loss 0.08122218 ; test accuracy 0.9743
epoch 8 : train loss 